In [4]:
from fear_greed_index import get_index
from constants import FEAR_GREED_INDEX_URL
URL = FEAR_GREED_INDEX_URL
df_index = get_index(URL, limit=30, format="json")
df_index

/home/geraltwolf/CryptoDataAnalysis/crypto_dashboard/functions.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '6 hours and 40 minutes' has dtype incompatible with timedelta64[ns], please explicitly cast to a compatible dtype first.
  res.iloc[0] = formatted


,value,value_classification,time_until_update,date
0,37,Fear,6 hours and 40 minutes,2025-11-02
1,33,Fear,NaT,2025-11-01
2,29,Fear,NaT,2025-10-31
3,34,Fear,NaT,2025-10-30
4,51,Neutral,NaT,2025-10-29
5,50,Neutral,NaT,2025-10-28
6,51,Neutral,NaT,2025-10-27
7,40,Fear,NaT,2025-10-26
8,37,Fear,NaT,2025-10-25
9,30,Fear,NaT,2025-10-24


In [15]:
def get_index_trend(df):
    
    value = df.iloc[0]['value_classification']
    
    if value == "Fear" or value == "Extreme Fear":
        ef_list = []
        for i in range(21):
            if df_index.iloc[i]['value'] >= 0 and df_index.iloc[i]['value'] <= 47:
                ef_list.append(i)
        if len(ef_list) >= 18:
            return "Long-term trend is stable"
        else:
            return "Long-term trend is unstable"
    
    if value == "Greed" or value == "Extreme Greed":
        eg_list = []
        for i in range(21):
            if df_index.iloc[i]['value'] >= 55 and df_index.iloc[i]['value'] <= 100:
                eg_list.append(i)
        if len(eg_list) >= 18:
            return "Long-term trend is stable"
        else:
            return "Long-term trend is unstable"

In [5]:
value = df_index.iloc[0:10]['value']
value

0    37
1    33
2    29
3    34
4    51
5    50
6    51
7    40
8    37
9    30
Name: value, dtype: int64

In [14]:
value = df_index.iloc[0]['value_classification']
if value == "Fear" or value == "Extreme Fear":
    ef_list = []
    for i in range(21):
        if df_index.iloc[i]['value'] >= 0 and df_index.iloc[i]['value'] <= 46:
            ef_list.append(i)
if len(ef_list) >= 18:
    print("Long-term trend is stable")
else:
    print("Long-term trend is unstable")

Long-term trend is stable


In [12]:
ef_list

[0, 1, 2, 3, 7, 8, 9, 10]

In [13]:
len(ef_list)

8

In [6]:
from inflation import get_cpi
from functions import get_historical_inflation
cpi = get_cpi()
new_df = get_historical_inflation(cpi)
new_df

In [ ]:
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from fear_greed_index import get_index
from constants import FEAR_GREED_INDEX_URL
URL = FEAR_GREED_INDEX_URL
from functions import create_gauge, get_index_recommendation, traffic_lights, get_recommendation
from functions import get_index_trend
from stockmarket import get_raw_stockmarket_data, get_yearly_stockmarket_trend
from stockmarket import get_montly_stockmarket_trend, get_yearly_stockmarket_data_for_dashboard
from inflation import get_cpi, get_inflation

def main():
    st.set_page_config(layout='wide')

    # Add back the CSS for centering text and button animation
    st.markdown("""
    <style>
    /* Center all text in the middle column */
    [data-testid="column"]:nth-child(2) {
        text-align: center !important;
    }
    
    /* Make sure all headers in the middle column are centered */
    [data-testid="column"]:nth-child(2) h1,
    [data-testid="column"]:nth-child(2) h2,
    [data-testid="column"]:nth-child(2) h3,
    [data-testid="column"]:nth-child(2) h4,
    [data-testid="column"]:nth-child(2) h5,
    [data-testid="column"]:nth-child(2) h6 {
        text-align: center !important;
    }
    
    /* Button pulsating animation */
    .stButton > button {
        animation: pulse 2s infinite !important;
        background-color: #FF4B4B !important;
        color: white !important;
        border: none !important;
        font-size: 20px !important;
        font-weight: bold !important;
        padding: 15px 30px !important;
        border-radius: 10px !important;
        width: 100% !important;
    }
    
    @keyframes pulse {
        0% { transform: scale(1); box-shadow: 0 0 0 0 rgba(255, 75, 75, 0.7); }
        70% { transform: scale(1.05); box-shadow: 0 0 0 15px rgba(255, 75, 75, 0); }
        100% { transform: scale(1); box-shadow: 0 0 0 0 rgba(255, 75, 75, 0); }
    }
    </style>
    """, unsafe_allow_html=True)

    if "page" not in st.session_state:
        st.session_state.page = 'start'

    if st.session_state.page == "start":
        col1, col2, col3 = st.columns([1, 6, 1])
        
        with col2:
            current_datetime = datetime.now()
            today = current_datetime.strftime("%B %d, %Y")
            st.subheader(f"Today is {today}")

            st.subheader("Crypto Fear & Greed Index is:")
            df = get_index(URL, limit=1, format="json")
            index = df.iloc[0]['value_classification']
            st.subheader(f"{index}")

            current_value = df.iloc[0]['value']
            fig_gauge = create_gauge(current_value)
            st.plotly_chart(fig_gauge, use_container_width=True)

            index_recommendation = get_index_recommendation(current_value)
            st.subheader(f"It tells you to {index_recommendation}")
            st.subheader("But should you? 🤔")

            # Simplified button centering - no nested columns needed
            if st.button("Verify F&G Index", key='verify_button'):
                st.session_state.page = "analysis"
                st.rerun()

    elif st.session_state.page == "analysis":
        # ... rest of your analysis page code remains exactly the same ...
        df_index = get_index(URL, limit=30, format="json")
        lt_trend = get_index_trend(df_index)
        raw_stockmarket = get_raw_stockmarket_data()
        df_stockmarket = get_montly_stockmarket_trend(raw_stockmarket)
        cpi = get_cpi()
        df_inflation = get_inflation(cpi)
        recommendation = get_recommendation(df_index, lt_trend, df_stockmarket, df_inflation)
        traffic_lights(recommendation)
        st.markdown(f"{recommendation}")
        st.markdown("Because:")

        #Initialize main columns:
        col1, col2, col3 = st.columns(3)

        #col1 - F&G Index:
        with col1:
            st.subheader(f"📈 Fear & Greed index:")
            st.write(f"Long-term trend is {lt_trend}")
                
            # Line chart
            df_fg_index = get_index(URL, timeout = 10, limit=30, format = 'json')
            fig_line = px.line(
                df_fg_index,
                x='date', 
                y='value',
                title=f"Fear & Greed Index - Last 30 Days",
                labels={'date': 'Date', 'value': 'Index Value'},
                color_discrete_sequence=["#1fb42b"]
            )
            
            # Add horizontal lines for different zones
            ap = 'top left'
            af = dict(size=16, family='Arial')
            fig_line.add_hline(y=25,
                               line_dash="dash",
                               line_color="red",
                               annotation_text="Extreme Fear",
                               annotation_position=ap,
                               annotation_font=af)
            fig_line.add_hline(y=45,
                               line_dash="dash",
                               line_color="orange",
                               annotation_text="Fear",
                               annotation_position=ap,
                               annotation_font=af)
            fig_line.add_hline(y=55,
                               line_dash="dash", 
                               line_color="yellow",
                               annotation_text="Neutral",
                               annotation_position=ap,
                               annotation_font=af)
            fig_line.add_hline(y=75,
                               line_dash="dash",
                               line_color="lightgreen",
                               annotation_text="Greed",
                               annotation_position=ap,
                               annotation_font=af)
            fig_line.add_hline(y=100,
                               line_dash="dash",
                               line_color="green",
                               annotation_text="Extreme Greed",
                               annotation_position=ap,
                               annotation_font=af)
            fig_line.update_layout(
                height=500,
                xaxis_title="Date",
                yaxis_title="Index Value (0-100)",
                yaxis=dict(range=[0, 100])
            )
            st.plotly_chart(fig_line, use_container_width=True)
        
        #col2 - Stockmarket
        with col2:
            raw_sm = get_raw_stockmarket_data()
            sm = get_yearly_stockmarket_data_for_dashboard(raw_sm)
            curr_sm = sm.iloc[-1]['stockmarket_value']
            monthly_sm = get_montly_stockmarket_trend(raw_sm)
            yearly_sm = get_yearly_stockmarket_trend(raw_sm)
            month_sm = sm.iloc[-25]['stockmarket_value']
            year_sm = sm.iloc[0]['stockmarket_value']

            #calculate montly rise / fall:
            monthly_change = (
                (sm.iloc[-1]['stockmarket_value'] - sm.iloc[-25]['stockmarket_value'])
                / sm.iloc[-25]['stockmarket_value']
                ) * 100
            
            #calculate yearly rise / fall:
            yearly_change = (
                (sm.iloc[-1]['stockmarket_value'] - sm.iloc[0]['stockmarket_value'])
                / sm.iloc[0]['stockmarket_value']
                ) * 100

            st.header("👩‍💼 Stock market")

            sm_col1, sm_col2, sm_col3 = st.columns(3)
            with sm_col1:
                st.metric('Current stock market value', f'{curr_sm:.2f}')
            with sm_col2:
                st.metric('Stock market value a month ago', f'{month_sm:.2f}')
            with sm_col3:
                st.metric('Stock market value a year ago', f'{year_sm:.2f}')  # Fixed label

            sm1_col1, sm1_col2 = st.columns(2)

            with sm1_col1:
                monthly_trend_value = monthly_sm.iloc[0]['stockmarket']
                monthly_direction = 'normal' if monthly_trend_value == 'Rising' else 'inverse' if monthly_trend_value == 'Falling' else 'off'
            
                st.metric(
                    label = 'Monthly stock market trend',
                    value=monthly_trend_value,
                    delta = f"{monthly_change:.2f}%",
                    delta_color=monthly_direction)
            
            with sm1_col2:
                yearly_trend_value = yearly_sm.iloc[0]['stockmarket']
                yearly_direction = 'normal' if yearly_trend_value == 'Rising' else 'inverse' if yearly_trend_value == 'Falling' else 'off'  # Fixed variable
            
                st.metric(
                    label = 'Yearly stock market trend',
                    value=yearly_trend_value,
                    delta = f"{yearly_change:.2f}%",
                    delta_color=yearly_direction)
                
            #Stockmarket long-term trend
            st.subheader('Stock market value distribution over the year')
            st.area_chart(sm.set_index('date')['stockmarket_value'])
        
        #col3 - Inflation
        with col3:
            #Inflation
            cpi = get_cpi()
            inflation = get_inflation(cpi)
            current_inflation = inflation.iloc[0]['current_inflation']
            inflation_growth = inflation.iloc[0]['inflation_growth']
            inflation_estimate = inflation.iloc[0]['inflation_estimate']
            st.header("💸 Inflation")

            inf_col1, inf_col2, inf_col3 = st.columns(3)
            with inf_col1:
                st.metric(label='Current Inflation', value = f"{current_inflation}%")
            
            with inf_col2:
                st.metric(label='Inflation growth', value = f"{inflation_growth}%")

            with inf_col3:
                st.metric(label='Inflation estimate', value = f"{inflation_estimate}")

            #Long-term inflatioin trend
            cpi = get_cpi()
            plot_cpi = cpi.copy()
            plot_cpi = plot_cpi[:-2]
            plot_cpi.sort_values(by = 'date', ascending=True, inplace=True)
            st.subheader("Inflation trend over the year")
            fig, ax = plt.subplots(figsize=(12, 6))
            sns.barplot(data = plot_cpi, x = 'date_formatted', y = 'hist_inf_rate', ax=ax)
            plt.bar_label(ax.containers[0], fmt='%.1f%%')
            plt.ylabel('Monthly inflation rate')
            plt.xlabel('Date')
            plt.xticks(rotation = 45)
            plt.tight_layout()
            st.pyplot(fig)

if __name__ == "__main__":
    main()

In [ ]:
def create_gauge(value):
    """
    Create a gauge with smooth gradient
    """
    # Calculate color based on current value (red -> yellow -> green)
    if value <= 50:
        # Red to Yellow
        r = 255
        g = int(255 * (value / 50))
        b = 0
    else:
        # Yellow to Green
        r = int(255 * (1 - (value - 50) / 50))
        g = 255
        b = 0
    
    number_color = f'rgb({r}, {g}, {b})'
    
    # Create smooth gradient by adding many small steps
    steps = []
    for i in range(100):
        # Calculate color based on position (red -> yellow -> green)
        if i <= 50:
            # Red to Yellow
            r_step = 255
            g_step = int(255 * (i / 50))
            b_step = 0
        else:
            # Yellow to Green
            r_step = int(255 * (1 - (i - 50) / 50))
            g_step = 255
            b_step = 0
        color = f'rgb({r_step}, {g_step}, {b_step})'
        steps.append({'range': [i, i+1], 'color': color})
    
    fig = go.Figure(go.Indicator(
        mode = "gauge+number",
        value = value,
        domain = {'x': [0, 1], 'y': [0, 1]},
        number = {'font': {'size': 48, 'color': number_color},  # Use dynamic color here
                  'suffix': '',
                  'valueformat': 'd'},
        gauge = {
            'axis': {'range': [0, 100],
                     'showticklabels': False,
                     'ticks': '',
                     'tickwidth': 0},
            'bar': {'color': "rgba(0,0,0,0)"},
            'bgcolor': "white",
            'borderwidth': 0,
            'bordercolor': "gray",
            'steps': steps,
            'threshold': {
                'line': {'color': "black", 'width': 3},
                'thickness': 0.4,
                'value': value
            }
        }
    ))
    
    fig.update_layout(
        height=150,
        margin=dict(l=20, r=20, t=20, b=20),
        paper_bgcolor='dark blue',
        font={'family': 'Arial'}
    )
    return fig